In [1]:
import random
import pandas as pd
from chemplus import mol_df
from chemplus import gen3d
from rdkit import Chem
from joblib import Parallel, delayed
from tqdm import tqdm
from IPython.display import display, HTML
from rdkit import RDLogger
pd.set_option("display.max_colwidth", None)

def pretty_print(df):
    return HTML(df.to_html().replace("\\n","<br>"))

# Set parameters for generation

In [ ]:
smiles_csv_path = r"../../../pre-trained/protonated_prepared_passed.csv"
out_sdf_3d_path = r"../../../pre-trained/protonated_prepared_passed_3D_2026-08-06_22-39.sdf"
out_log_csv_path = r"../../../pre-trained/sdf_log.csv"
conformers_num = 3
processors_num = 6
smiles_column = "SMILES"
name_column = "Mol ID"

# Generate and save 3D mols
This code saves logs for all molecules and stores an SDF file with successfully generated 3D molecules.

In [3]:
RDLogger.DisableLog('rdApp.*')
df = pd.read_csv(smiles_csv_path)
if df[name_column].duplicated().sum():
    raise Exception(f"The name_column should not contain duplicates. Found {df[name_column].duplicated().sum()} duplicates")
if df[smiles_column].duplicated().sum():
    df = df.drop_duplicates(subset=[name_column])
print(f"Number of SMILES: {df.shape[0]}")
df["Mol"] = df[smiles_column].apply(Chem.MolFromSmiles)
if conformers_num > 1:
    df["Conformer"] = [list(map(str, range(1, conformers_num + 1)))]*df.shape[0]
    df = df.explode("Conformer", ignore_index=True)
    df[name_column] = df[name_column] + "_conf-" + df["Conformer"]
    df = df.drop(columns="Conformer")

def try_to_gen_3d(mol, attempts=3):
    # Safely handle None molecules if RDKit failed to parse the SMILES earlier
    if mol is None:
        return (None, "Invalid molecule (None)")
        
    seed_list = random.sample(range(1, 2**16), attempts)
    log = ""
    
    for idx, seed in enumerate(seed_list):
        try:
            result = gen3d.generate_3d(mol, seed=seed)
            if result[0] is not None:
                return result
            log += f"Attempt {idx+1} failed:\n{result[1]}\n"
        except Exception as e:
            # Catch the RDKit C++ RuntimeError (UFF parameter failure)
            log += f"Attempt {idx+1} exception: {str(e)}\n"
            
    return (None, log.strip())

mols_3d = Parallel(n_jobs=processors_num, verbose=0, backend="loky")(delayed(try_to_gen_3d)(mol) for mol in tqdm(df["Mol"].values))
df["Mol 3D"], df["gen3D log"] = tuple(zip(*mols_3d))
print(f"Number of successfully generated conformers: {df['Mol 3D'].notna().sum()}")
print(f"Number of unsuccessfully generated conformers: {df['Mol 3D'].isna().sum()}")
df[[name_column, "gen3D log"]].to_csv(out_log_csv_path, index=False)
mol_df.df_to_sdf(df[~df["Mol 3D"].isna()], out_sdf_3d_path, mol_column="Mol 3D", name_column=name_column)

Number of SMILES: 88529


100%|██████████| 265587/265587 [1:59:07<00:00, 37.16it/s]  


Number of successfully generated conformers: 265566
Number of unsuccessfully generated conformers: 21


# Check unsuccessfully generated 3D mols

In [ ]:
pretty_print(df[df["Mol 3D"].isna()][[name_column, "Mol", "gen3D log"]])